# Gaia DR3 Variable Stars: Feature Engineering

## Imports and Configuration

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.timeseries import LombScargle
from astropy_healpix import HEALPix
from astropy import units as u
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

for path in (root / "src", root):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gaia_variable_stars.config import (
    SOURCE_FILE, CLEAN_FILE, PHOTOMETRY_FILE,
    FEATURE_FILE, CLASSES, MIN_POINTS, SEED
)

pd.set_option("display.max_columns", 100)
MIN_PERIOD = 0.1
MAX_PERIOD = 1000.0
GRID_SIZE = 5
MAX_DRIFT = 0.25
N_PEAKS = 5
N_HARMONICS = 4
CORR_LIMIT = 0.90
USE_WEIGHTS = False

FE_FEATURE_FILE = FEATURE_FILE.with_name("gaia_features_fe.csv")
FE_VALIDATION_FILE = FEATURE_FILE.with_name("gaia_period_validation_fe.csv")
FE_ORACLE_FILE = FEATURE_FILE.with_name("gaia_features_fe_oracle.csv")

## Load Data

In [2]:
sources = pd.read_csv(SOURCE_FILE)
photometry = pd.read_csv(CLEAN_FILE, dtype={"transit_id": "string"})

source_cols = [
    "source_id", "class_name", "bp_rp_color", "ecl_lat",
    "ra", "dec", "g_mean_magnitude", "gaia_period",
    "gaia_period_origin"
]
photo_cols = [
    "source_id", "g_transit_time", "g_transit_mag",
    "g_transit_flux", "g_transit_flux_error", "variability_flag_g_reject"
]
missing = (set(source_cols) - set(sources)) | (set(photo_cols) - set(photometry))
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if sources["source_id"].duplicated().any():
    raise ValueError("Duplicate source IDs.")
if not sources["class_name"].isin(CLASSES).all():
    raise ValueError("Unexpected variable-star class.")
if not set(photometry["source_id"]).issubset(set(sources["source_id"])):
    raise ValueError("Photometry contains unknown source IDs.")

for col in ["bp_rp_color", "ecl_lat", "ra", "dec", "gaia_period", "g_mean_magnitude"]:
    sources[col] = pd.to_numeric(sources[col], errors="coerce")

print("Sources:", len(sources))
print("Clean observations:", len(photometry))
display(sources["class_name"].value_counts().reindex(CLASSES, fill_value=0))

Sources: 2000
Clean observations: 85243


class_name
RR Lyrae                500
Cepheid                 500
Eclipsing Binary        500
Long-Period Variable    500
Name: count, dtype: int64

## Prepare Photometry

In [3]:
observations = photometry.copy()
for col in ["g_transit_time", "g_transit_mag", "g_transit_flux", "g_transit_flux_error"]:
    observations[col] = pd.to_numeric(observations[col], errors="coerce")

valid = (
    np.isfinite(observations[["g_transit_time", "g_transit_mag", "g_transit_flux"]]).all(axis=1)
    & observations["g_transit_flux"].gt(0)
    & observations["variability_flag_g_reject"].astype("string")
      .str.strip().str.lower().isin(["false", "0"])
)
if not valid.all():
    raise ValueError("CLEAN_FILE contains invalid or rejected observations.")
if observations.duplicated(["source_id", "g_transit_time"]).any():
    raise ValueError("Duplicate source timestamps in CLEAN_FILE.")

flux = observations["g_transit_flux"]
error = observations["g_transit_flux_error"]
observations["magnitude_error"] = (
    2.5 / np.log(10) * error / flux
).where(np.isfinite(error) & error.gt(0))
observations = observations.sort_values(
    ["source_id", "g_transit_time"]
).reset_index(drop=True)

print("Valid observations:", len(observations))
print("Valid magnitude errors:", observations["magnitude_error"].notna().sum())

Valid observations: 85243
Valid magnitude errors: 85243


## Period Search and Fourier Fits

In [4]:
def period_search(time, mag, errors=None):
    baseline = np.ptp(time)
    max_period = min(MAX_PERIOD, baseline / 2)
    searchable = len(time) >= MIN_POINTS and np.unique(time).size >= MIN_POINTS and np.std(mag) > 0
    if not searchable or max_period <= MIN_PERIOD:
        return None

    weighted = USE_WEIGHTS and errors is not None and np.isfinite(errors).all() and (errors > 0).all()
    periodogram = LombScargle(time - time.min(), mag, dy=errors if weighted else None,
                              normalization="standard")
    frequency, powers = periodogram.autopower(
        minimum_frequency=1 / max_period, maximum_frequency=1 / MIN_PERIOD,
        samples_per_peak=GRID_SIZE,
    )
    valid = (frequency >= 1 / max_period) & (frequency <= 1 / MIN_PERIOD)
    frequency, powers = frequency[valid], powers[valid]
    finite = np.isfinite(powers)
    if not finite.any():
        raise ValueError("No finite Lomb–Scargle powers.")

    available = np.where(finite, powers, -np.inf).copy()
    peaks = []
    for _ in range(N_PEAKS):
        peak = int(np.argmax(available))
        if not np.isfinite(available[peak]):
            break
        peaks.append((float(1 / frequency[peak]), float(powers[peak])))
        available[np.abs(frequency - frequency[peak]) <= 1 / baseline] = -np.inf

    best_period, best_power = peaks[0]
    result = {"periodogram": periodogram, "peaks": peaks, "period": best_period, "power": best_power,
              "search_max_period": max_period}

    periods = 1 / frequency
    for name, select in {
        "short": periods < 1, "medium": (periods >= 1) & (periods <= 30),
        "long": periods > 30,
    }.items():
        indices = np.flatnonzero(select & finite)
        chosen = indices[np.argmax(powers[indices])] if indices.size else None
        result[f"best_{name}_period"] = float(periods[chosen]) if chosen is not None else np.nan
        result[f"best_{name}_power"] = float(powers[chosen]) if chosen is not None else np.nan

    result["peak_ratio_2_1"] = peaks[1][1] / best_power if len(peaks) > 1 and best_power > 0 else np.nan
    median_power = np.median(powers[finite])
    result["peak_to_median"] = best_power / median_power if median_power > 0 else np.nan
    result["near_search_boundary"] = bool(
        1 / best_period - 1 / max_period <= 1 / baseline
        or 1 / MIN_PERIOD - 1 / best_period <= 1 / baseline
    )
    return result


def fourier_fit(time, mag, period, harmonics=(1, 2, 3, 4)):
    if not np.isfinite(period) or period <= 0:
        return None
    phase = 2 * np.pi * ((time - time.min()) / period % 1)
    columns = [np.ones(len(time))]
    for k in harmonics:
        columns.extend([np.cos(k * phase), np.sin(k * phase)])
    design = np.column_stack(columns)
    if len(time) <= design.shape[1] or np.linalg.matrix_rank(design) != design.shape[1]:
        return None
    coeff = np.linalg.lstsq(design, mag, rcond=None)[0]
    rss = np.sum((mag - design @ coeff) ** 2)
    bic = len(time) * np.log(max(rss / len(time), np.finfo(float).tiny)) + len(coeff) * np.log(len(time))
    amps = {k: np.hypot(*coeff[1 + 2 * i:3 + 2 * i]) for i, k in enumerate(harmonics)}
    phases = {k: np.arctan2(-coeff[2 + 2 * i], coeff[1 + 2 * i]) for i, k in enumerate(harmonics)}
    result = {"coeff": coeff, "bic": bic, "amps": amps, "phases": phases}
    if {1, 2, 3}.issubset(amps) and amps[1] > 0:
        result["R21"] = amps[2] / amps[1]
        result["R31"] = amps[3] / amps[1]
        result["phi21"] = (phases[2] - 2 * phases[1] + np.pi) % (2 * np.pi) - np.pi
        result["phi31"] = (phases[3] - 3 * phases[1] + np.pi) % (2 * np.pi) - np.pi
    return result


def resolve_period(time, mag, period, max_period):
    naive = fourier_fit(time, mag, period)
    result = {"period_resolved": period, "delta_bic": np.nan,
              "odd_harmonic_fraction": np.nan, "odd_even_depth_difference": np.nan}
    if naive is None:
        return result, naive, naive

    even = fourier_fit(time, mag, 2 * period, harmonics=(2, 4))
    full = fourier_fit(time, mag, 2 * period)
    if even is None or full is None:
        return result, naive, naive

    result["delta_bic"] = even["bic"] - full["bic"]
    amps = full["amps"]
    norm = np.sqrt(sum(amp ** 2 for amp in amps.values()))
    if norm > 0:
        result["odd_harmonic_fraction"] = np.hypot(amps[1], amps[3]) / norm

    grid = np.arange(1024) / 1024
    model = np.full(len(grid), full["coeff"][0])
    for k in range(1, N_HARMONICS + 1):
        model += full["coeff"][2 * k - 1] * np.cos(2 * np.pi * k * grid)
        model += full["coeff"][2 * k] * np.sin(2 * np.pi * k * grid)
    peak = int(np.argmax(model))
    result["odd_even_depth_difference"] = abs(model[peak] - model[(peak + 512) % 1024])

    if result["delta_bic"] > 2:
        result["period_resolved"] = 2 * period
        return result, naive, full
    return result, naive, naive

## Period-Free Features

In [5]:
def timescale_features(time, mag, errors):
    gaps = np.diff(time)
    delta = np.diff(mag)
    visit = (gaps > 0) & (gaps <= 1)
    pair_error = errors[:-1] ** 2 + errors[1:] ** 2
    adjusted = np.maximum(delta ** 2 - pair_error, 0)
    slopes = np.sqrt(adjusted[visit & np.isfinite(adjusted)]) / gaps[visit & np.isfinite(adjusted)]

    left, right = np.triu_indices(len(time), k=1)
    lags = time[right] - time[left]
    squared = (mag[right] - mag[left]) ** 2
    noise = errors[left] ** 2 + errors[right] ** 2
    usable = np.isfinite(noise)

    short = usable & (lags >= 0.05) & (lags <= 0.3)
    long = usable & (lags > 6)
    sf_short = float(np.mean((squared - noise)[short])) if short.any() else np.nan
    sf_long = float(np.mean((squared - noise)[long])) if long.any() else np.nan

    return {
        "sf_short": sf_short,
        "sf_long": sf_long,
        "sf_ratio": sf_short / sf_long if np.isfinite(sf_short) and np.isfinite(sf_long) and sf_long > 0 else np.nan,
        "max_visit_slope": float(np.max(slopes)) if len(slopes) else np.nan,
    }


def stetson_j(time, mag, errors):
    valid = np.isfinite(errors) & (errors > 0)
    if valid.sum() < 2:
        return np.nan
    normalized = np.full(len(mag), np.nan)
    n = valid.sum()
    mean = np.average(mag[valid], weights=1 / errors[valid] ** 2)
    normalized[valid] = np.sqrt(n / (n - 1)) * (mag[valid] - mean) / errors[valid]
    pairs = (np.diff(time) > 0) & (np.diff(time) <= 0.3)
    product = normalized[:-1] * normalized[1:]
    selected = product[pairs & np.isfinite(product)]
    return float(np.mean(np.sign(selected) * np.sqrt(np.abs(selected)))) if len(selected) else np.nan

## Extract Features

In [6]:
records = []
search_failures = []
light_curves = {}

for source_id, group in observations.groupby("source_id", sort=False):
    time = group["g_transit_time"].to_numpy(dtype=float)
    mag = group["g_transit_mag"].to_numpy(dtype=float)
    errors = group["magnitude_error"].to_numpy(dtype=float)
    light_curves[source_id] = (time, mag, errors)

    gaps = np.diff(time)
    visit_gaps = gaps[gaps > 1]
    n_visits = int(1 + len(visit_gaps))
    amplitude = group["g_transit_mag"].quantile(0.95) - group["g_transit_mag"].quantile(0.05)
    median_error = group["magnitude_error"].median()
    record = {
        "source_id": source_id,
        "magnitude_mean": mag.mean(),
        "magnitude_std": mag.std(ddof=1) if len(mag) > 1 else np.nan,
        "magnitude_skew": group["g_transit_mag"].skew(),
        "magnitude_kurt": group["g_transit_mag"].kurt(),
        "magnitude_mad": np.median(np.abs(mag - np.median(mag))),
        "amplitude": amplitude,
        "median_error": median_error,
        "amplitude_ratio": amplitude / median_error if np.isfinite(median_error) and median_error > 0 else np.nan,
        "n_epochs": len(time),
        "time_span": np.ptp(time),
        "median_gap": np.median(gaps) if len(gaps) else np.nan,
        "short_gap_fraction": np.mean(gaps < 1) if len(gaps) else np.nan,
        "n_visits": n_visits,
        "epochs_per_visit": len(time) / n_visits,
        "median_visit_gap": np.median(visit_gaps) if len(visit_gaps) else np.nan,
        "visit_gap_cv": np.std(visit_gaps, ddof=1) / np.mean(visit_gaps) if len(visit_gaps) > 1 else np.nan,
        "searchable": False,
        "period_estimate": np.nan,
        "period_resolved": np.nan,
        "log_period": np.nan,
        "power": np.nan,
        "half_power": np.nan,
        "double_power": np.nan,
        "half_power_ratio": np.nan,
        "double_power_ratio": np.nan,
        "fourier_amp": np.nan,
        "fourier_ratio": np.nan,
        "fourier_amp_resolved": np.nan,
        "fourier_ratio_resolved": np.nan,
        "delta_bic": np.nan,
        "odd_harmonic_fraction": np.nan,
        "odd_even_depth_difference": np.nan,
        "search_max_period": np.nan,
        "near_search_boundary": np.nan,
        "peak_ratio_2_1": np.nan,
        "peak_to_median": np.nan,
        "beyond_1sigma_fraction": np.mean(np.abs(mag - np.median(mag)) > mag.std()) if mag.std() > 0 else np.nan,
        "von_neumann_eta": np.mean(np.diff(mag) ** 2) / np.var(mag, ddof=1) if len(mag) > 1 and np.var(mag, ddof=1) > 0 else np.nan,
        "stetson_j": stetson_j(time, mag, errors),
    }
    record.update(timescale_features(time, mag, errors))

    for k in range(1, N_PEAKS + 1):
        record[f"peak_period_{k}"] = np.nan
        record[f"peak_power_{k}"] = np.nan
    for band in ["short", "medium", "long"]:
        record[f"best_{band}_period"] = np.nan
        record[f"best_{band}_power"] = np.nan
    for period_type in ["naive", "resolved"]:
        for field in ["R21", "phi21", "R31", "phi31"]:
            record[f"{field}_{period_type}"] = np.nan

    try:
        search = period_search(time, mag, errors)
        if search is not None:
            period = search["period"]
            power = search["power"]
            f = 1 / period
            record["searchable"] = True
            record["period_estimate"] = period
            record["log_period"] = np.log10(period)
            record["power"] = power
            # Deleted: grid-bound checks around half- and double-period power.
            record["half_power"] = float(search["periodogram"].power(2 * f))
            record["double_power"] = float(search["periodogram"].power(f / 2))
            record["half_power_ratio"] = record["half_power"] / power if power > 0 else np.nan
            record["double_power_ratio"] = record["double_power"] / power if power > 0 else np.nan
            record["search_max_period"] = search["search_max_period"]
            record["near_search_boundary"] = search["near_search_boundary"]
            for name in ["peak_ratio_2_1", "peak_to_median"]:
                record[name] = search[name]
            for name in ["short", "medium", "long"]:
                record[f"best_{name}_period"] = search[f"best_{name}_period"]
                record[f"best_{name}_power"] = search[f"best_{name}_power"]
            for k, (peak_period, peak_power) in enumerate(search["peaks"], 1):
                record[f"peak_period_{k}"] = peak_period
                record[f"peak_power_{k}"] = peak_power

            resolution, naive, resolved = resolve_period(time, mag, period, search["search_max_period"])
            record.update(resolution)
            if naive is not None:
                record["fourier_amp"] = naive["amps"].get(1, np.nan)
                record["fourier_ratio"] = naive.get("R21", np.nan)
            if resolved is not None:
                record["fourier_amp_resolved"] = resolved["amps"].get(1, np.nan)
                record["fourier_ratio_resolved"] = resolved.get("R21", np.nan)
            for label, fit in [("naive", naive), ("resolved", resolved)]:
                if fit is not None:
                    for field in ["R21", "phi21", "R31", "phi31"]:
                        record[f"{field}_{label}"] = fit.get(field, np.nan)

    except (ValueError, FloatingPointError, ZeroDivisionError, np.linalg.LinAlgError) as exc:
        search_failures.append((source_id, f"{type(exc).__name__}: {exc}"))
    records.append(record)

## Check Extraction

In [7]:
if search_failures:
    raise RuntimeError(f"Period searches failed for {len(search_failures)} sources: {search_failures[:5]}")

if len(records) != len(sources) or sum(r["n_epochs"] for r in records) != len(observations):
    raise RuntimeError("Extracted source or epoch counts do not match CLEAN_FILE.")

if any(r["searchable"] and (not np.isfinite(r["period_estimate"]) or not np.isfinite(r["power"])) for r in records):
    raise RuntimeError("Searchable sources have invalid period-search results.")

if not all(np.isfinite(r["amplitude"]) and r["amplitude"] >= 0 for r in records):
    raise RuntimeError("Invalid photometric amplitudes.")

bad_periods = [
    (r["source_id"], r["period_estimate"], r["search_max_period"], r["log_period"])
    for r in records if r["searchable"] and not (
        (r["period_estimate"] >= MIN_PERIOD or np.isclose(r["period_estimate"], MIN_PERIOD, rtol=1e-10, atol=0))
        and (r["period_estimate"] <= r["search_max_period"] or np.isclose(r["period_estimate"], r["search_max_period"], rtol=1e-10, atol=0))
        and np.isclose(np.log10(r["period_estimate"]), r["log_period"], rtol=0, atol=1e-10)
    )
]

if bad_periods:
    raise RuntimeError(f"Invalid period boundaries or log_period for {len(bad_periods)} sources: {bad_periods[:5]}")

print("Extracted Sources:", len(records))
print("Searchable Sources:", sum(r["searchable"] for r in records))

Extracted Sources: 2000
Searchable Sources: 1978


## Feature Tables

In [8]:
stars = sources[["source_id", "class_name", "bp_rp_color", "ecl_lat", "ra", "dec",
                 "g_mean_magnitude", "gaia_period", "gaia_period_origin"]].merge(
    pd.DataFrame(records), on="source_id", how="left", validate="one_to_one"
)
stars["abs_latitude"] = stars["ecl_lat"].abs()
stars["bp_rp_missing"] = stars["bp_rp_color"].isna().astype(int)

primary_cols = [
    "magnitude_mean", "magnitude_std", "magnitude_skew", "magnitude_kurt",
    "amplitude", "log_period", "power", "fourier_amp", "fourier_ratio",
    "bp_rp_color", "magnitude_mad", "median_error", "amplitude_ratio", "half_power", "double_power",
]
additional_cols = [
    "half_power_ratio", "double_power_ratio", "peak_ratio_2_1", "peak_to_median",
    "period_resolved", "delta_bic", "odd_harmonic_fraction", "odd_even_depth_difference",
    "fourier_amp_resolved", "fourier_ratio_resolved",
    "beyond_1sigma_fraction", "von_neumann_eta", "stetson_j", "sf_short", "sf_long",
    "sf_ratio", "max_visit_slope",
    *[f"peak_{kind}_{k}" for k in range(1, N_PEAKS + 1) for kind in ["period", "power"]],
    *[f"best_{band}_{kind}" for band in ["short", "medium", "long"] for kind in ["period", "power"]],
    *[f"{field}_{label}" for label in ["naive", "resolved"]
      for field in ["R21", "phi21", "R31", "phi31"]],
]
sampling_cols = [
    "n_epochs", "time_span", "median_gap", "short_gap_fraction",
    "n_visits", "epochs_per_visit", "median_visit_gap", "visit_gap_cv",
]
predictor_cols = primary_cols + additional_cols
stars[predictor_cols + sampling_cols] = stars[predictor_cols + sampling_cols].replace(
    [np.inf, -np.inf], np.nan
)

model_sources = stars["searchable"].fillna(False)
features = stars.loc[model_sources, ["source_id", "class_name", *predictor_cols]].copy()
ablation = stars.loc[model_sources, ["source_id", "class_name", *predictor_cols,
                                    *sampling_cols, "bp_rp_missing"]].copy()

excluded = {"gaia_period", "gaia_period_origin", "abs_latitude", "period_eligible",
            "period_status", "half_period_alias", "double_period_alias", "recovered"}
if excluded.intersection(features.columns):
    raise ValueError("Validation-only columns entered the classification predictors.")
if not all(pd.api.types.is_numeric_dtype(features[col]) for col in predictor_cols):
    raise ValueError("Nonnumeric predictors.")
if np.isinf(features[predictor_cols].to_numpy(dtype=float)).any():
    raise ValueError("Infinite predictors.")
if not features["source_id"].is_unique:
    raise ValueError("Duplicate model rows.")

ineligible = stars.loc[~model_sources, ["source_id", "class_name", "n_epochs"]]
print("Excluded from Classification:", len(ineligible))
display(ineligible.groupby("class_name").size().reindex(CLASSES, fill_value=0).rename("excluded"))

Excluded from Classification: 22


class_name
RR Lyrae                22
Cepheid                  0
Eclipsing Binary         0
Long-Period Variable     0
Name: excluded, dtype: int64

## Period Recovery

In [9]:
def period_match(estimate, reference, span):
    estimate, reference, span = np.broadcast_arrays(
        np.asarray(estimate, dtype=float),
        np.asarray(reference, dtype=float),
        np.asarray(span, dtype=float)
    )
    valid = (
        np.isfinite(estimate) & (estimate > 0)
        & np.isfinite(reference) & (reference > 0)
        & np.isfinite(span) & (span > 0)
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        drift = span * np.abs(1 / estimate - 1 / reference)
    return valid & (drift <= MAX_DRIFT)

estimate = stars["period_estimate"].to_numpy(dtype=float)
published = stars["gaia_period"].to_numpy(dtype=float)
span = stars["time_span"].to_numpy(dtype=float)
maximum = stars["search_max_period"].to_numpy(dtype=float)

eligible = (
    stars["searchable"].fillna(False).to_numpy(dtype=bool)
    & np.isfinite(published) & (published >= MIN_PERIOD)
    & (published <= maximum)
)

exact = eligible & period_match(estimate, published, span)
half = eligible & ~exact & period_match(estimate, published / 2, span)
double = eligible & ~exact & ~half & period_match(estimate, published * 2, span)

resolved_period = stars["period_resolved"].to_numpy(dtype=float)
stars["resolved_exact"] = eligible & period_match(resolved_period, published, span)

stars["period_eligible"] = eligible
stars["phase_coherent_recovery"] = exact
stars["half_period_alias"] = half
stars["double_period_alias"] = double
stars["period_status"] = np.select(
    [~eligible, exact, half, double],
    ["ineligible", "exact", "half", "double"], default="mismatch"
)
with np.errstate(divide="ignore", invalid="ignore"):
    stars["primary_relative_error"] = np.where(
        eligible, np.abs(estimate / published - 1), np.nan
    )
    stars["primary_phase_drift_cycles"] = np.where(
        eligible, span * np.abs(1 / estimate - 1 / published), np.nan
    )
stars["relative_period_accurate"] = (
    eligible & (stars["primary_relative_error"].to_numpy() <= 0.10)
)

for k in [1, 3, 5]:
    recovered = np.zeros(len(stars), dtype=bool)
    for j in range(1, k + 1):
        peak = stars[f"peak_period_{j}"].to_numpy(dtype=float)
        recovered |= period_match(peak, published, span)
    stars[f"recovery_at_{k}"] = eligible & recovered

recovery = pd.crosstab(
    stars.loc[eligible, "class_name"], stars.loc[eligible, "period_status"]
).reindex(index=CLASSES, columns=["exact", "half", "double", "mismatch"], fill_value=0)

display(recovery)
display(
    stars.loc[eligible].groupby("class_name")[
        ["recovery_at_1", "recovery_at_3", "recovery_at_5"]
    ].mean().reindex(CLASSES).round(3)
)

period_status,exact,half,double,mismatch
class_name,,,,
RR Lyrae,392,1,0,85
Cepheid,495,0,0,5
Eclipsing Binary,12,366,2,120
Long-Period Variable,269,0,0,132


,recovery_at_1,recovery_at_3,recovery_at_5
class_name,,,
RR Lyrae,0.820,0.921,0.948
Cepheid,0.990,0.996,0.996
Eclipsing Binary,0.024,0.054,0.062
Long-Period Variable,0.671,0.805,0.863


## Sky Position and Observation Quality

In [10]:
healpix = HEALPix(nside=16, order="ring", frame="icrs")
sky_cell = np.full(len(stars), -1, dtype=int)
valid_sky = stars["ra"].between(0, 360) & stars["dec"].between(-90, 90)
sky_cell[valid_sky] = healpix.lonlat_to_healpix(
    stars.loc[valid_sky, "ra"].to_numpy() * u.deg,
    stars.loc[valid_sky, "dec"].to_numpy() * u.deg
)
stars["sky_cell"] = sky_cell

raw = pd.read_csv(
    PHOTOMETRY_FILE,
    usecols=lambda col: col in {
        "source_id", "band", "variability_flag_g_reject", "rejected_by_variability"
    }
)
if "band" in raw:
    raw = raw.loc[raw["band"].astype("string").str.upper().eq("G")]

flag_col = (
    "variability_flag_g_reject" if "variability_flag_g_reject" in raw
    else "rejected_by_variability"
)
flags = raw[flag_col].astype("string").str.strip().str.lower()
raw["rejected"] = np.select(
    [flags.isin(["true", "1"]), flags.isin(["false", "0"])],
    [1.0, 0.0], default=np.nan
)
rejection = raw.groupby("source_id")["rejected"].mean()
stars["gaia_rejection_fraction"] = stars["source_id"].map(rejection)

print("Sky cells:", stars.loc[stars["sky_cell"].ge(0), "sky_cell"].nunique())
print("Sources with rejection data:", stars["gaia_rejection_fraction"].notna().sum())

Sky cells: 169
Sources with rejection data: 2000


In [11]:
flow = pd.DataFrame({
    "selected": sources["class_name"].value_counts(),
    "retrieved_g_band": sources.loc[sources["source_id"].isin(raw["source_id"]), "class_name"].value_counts(),
    "clean_photometry": sources.loc[sources["source_id"].isin(observations["source_id"]), "class_name"].value_counts(),
    "searchable_modeled": features["class_name"].value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)

bad = stars.loc[~model_sources]
reason = np.select(
    [bad["n_epochs"].lt(MIN_POINTS), (bad["time_span"] / 2).le(MIN_PERIOD)],
    ["n_epochs < MIN_POINTS", "baseline/2 <= MIN_PERIOD"],
    default="constant magnitude or too few unique times",
)
flow = flow.join(pd.crosstab(bad["class_name"], reason).reindex(CLASSES, fill_value=0))
flow.to_csv(FEATURE_FILE.with_name("gaia_cohort_flow.csv"))
display(flow)

,selected,retrieved_g_band,clean_photometry,searchable_modeled,n_epochs < MIN_POINTS
class_name,,,,,
RR Lyrae,500,500,500,478,22
Cepheid,500,500,500,500,0
Eclipsing Binary,500,500,500,500,0
Long-Period Variable,500,500,500,500,0


## Period Validation

In [12]:
with np.errstate(divide="ignore", invalid="ignore"):
    stars["n_cycles"] = stars["time_span"] / stars["gaia_period"]
    stars["epochs_per_cycle"] = stars["n_epochs"] / stars["n_cycles"]
    stars["visits_per_cycle"] = stars["n_visits"] / stars["n_cycles"]

validation_cols = [
    "source_id", "class_name", "gaia_period", "gaia_period_origin",
    "period_estimate", "period_resolved", "g_mean_magnitude",
    "ecl_lat", "abs_latitude", "sky_cell", "gaia_rejection_fraction",
    "n_cycles", "epochs_per_cycle", "visits_per_cycle",
    *sampling_cols, "search_max_period", "near_search_boundary",
    "period_eligible", "primary_relative_error",
    "primary_phase_drift_cycles", "relative_period_accurate",
    "phase_coherent_recovery", "half_period_alias", "double_period_alias",
    "resolved_exact",
    "period_status", "recovery_at_1", "recovery_at_3", "recovery_at_5",
    *[f"peak_{kind}_{k}" for k in range(1, N_PEAKS + 1) for kind in ["period", "power"]]
]
period_validation = (
    stars[validation_cols].replace([np.inf, -np.inf], np.nan).copy()
)

display(
    period_validation.loc[period_validation["period_eligible"]]
    .groupby("class_name")
    .agg(
        stars=("source_id", "size"),
        median_cycles=("n_cycles", "median"),
        median_epochs=("n_epochs", "median"),
        recovery=("phase_coherent_recovery", "mean")
    ).reindex(CLASSES).round(3)
)

,stars,median_cycles,median_epochs,recovery
class_name,,,,
RR Lyrae,478,1967.233,37.0,0.820
Cepheid,500,305.746,39.0,0.990
Eclipsing Binary,500,2243.189,40.0,0.024
Long-Period Variable,401,5.183,42.0,0.671


## Oracle Features

In [13]:
oracle_records = []
for _, star in stars.loc[eligible].iterrows():
    time, mag, errors = light_curves[star["source_id"]]
    period = float(star["gaia_period"])
    maximum = float(star["search_max_period"])
    search = LombScargle(time - time.min(), mag, normalization="standard")
    power = float(search.power(1 / period))
    half_power = float(search.power(2 / period))
    double_power = float(search.power(1 / (2 * period)))
    resolution, naive, resolved = resolve_period(time, mag, period, maximum)
    row = {
        "source_id": star["source_id"], "class_name": star["class_name"],
        "gaia_period": period, "log_period": np.log10(period), "power": power,
        "half_power": half_power, "double_power": double_power,
        "half_power_ratio": half_power / power if power > 0 else np.nan,
        "double_power_ratio": double_power / power if power > 0 else np.nan,
        "period_resolved": resolution["period_resolved"],
        "delta_bic": resolution["delta_bic"],
        "odd_harmonic_fraction": resolution["odd_harmonic_fraction"],
        "odd_even_depth_difference": resolution["odd_even_depth_difference"],
        "fourier_amp": naive["amps"].get(1, np.nan) if naive else np.nan,
        "fourier_ratio": naive.get("R21", np.nan) if naive else np.nan,
        "fourier_amp_resolved": resolved["amps"].get(1, np.nan) if resolved else np.nan,
        "fourier_ratio_resolved": resolved.get("R21", np.nan) if resolved else np.nan,
    }
    for label, fit in [("naive", naive), ("resolved", resolved)]:
        for field in ["R21", "phi21", "R31", "phi31"]:
            row[f"{field}_{label}"] = fit.get(field, np.nan) if fit else np.nan
    oracle_records.append(row)

oracle = pd.DataFrame(oracle_records).replace([np.inf, -np.inf], np.nan)


print("Oracle sources:", len(oracle))
display(oracle.groupby("class_name").size().reindex(CLASSES, fill_value=0))

Oracle sources: 1879


class_name
RR Lyrae                478
Cepheid                 500
Eclipsing Binary        500
Long-Period Variable    401
dtype: int64

## Check Features

In [14]:
diagnostics = pd.DataFrame({
    "feature": predictor_cols,
    "missing": features[predictor_cols].isna().sum().to_numpy(),
    "missing_pct": 100 * features[predictor_cols].isna().mean().to_numpy()
})
display(diagnostics.sort_values("missing_pct", ascending=False).round(2))

corr = features[predictor_cols].corr(method="spearman").abs()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
display(pairs[pairs.ge(CORR_LIMIT)].sort_values(ascending=False).rename("abs_corr").to_frame())

if features["source_id"].duplicated().any():
    raise ValueError("Duplicate classification sources.")
if not features["source_id"].isin(stars.loc[stars["searchable"], "source_id"]).all():
    raise ValueError("Ineligible sources entered classification.")
if np.isinf(features[predictor_cols].to_numpy(dtype=float)).any():
    raise ValueError("Infinite classification predictors.")
if set(oracle["source_id"]) != set(stars.loc[stars["period_eligible"], "source_id"]):
    raise ValueError("Oracle coverage differs from eligible sources.")

print("Complete predictor rows:", features[predictor_cols].notna().all(axis=1).sum())

,feature,missing,missing_pct
9,bp_rp_color,39,1.97
30,sf_ratio,30,1.52
42,best_short_period,0,0.00
31,max_visit_slope,0,0.00
32,peak_period_1,0,0.00
33,peak_power_1,0,0.00
34,peak_period_2,0,0.00
35,peak_power_2,0,0.00
36,peak_period_3,0,0.00
37,peak_power_3,0,0.00


abs_corr
fourier_ratio          R21_naive           1.000000
fourier_ratio_resolved R21_resolved        1.000000
power                  peak_power_1        1.000000
log_period             peak_period_1       1.000000
                       period_resolved     0.988947
period_resolved        peak_period_1       0.988947
peak_power_4           peak_power_5        0.982608
magnitude_std          amplitude           0.979082
double_power           double_power_ratio  0.968426
peak_power_3           peak_power_4        0.966607
magnitude_std          sf_long             0.962214
half_power             half_power_ratio    0.952338
peak_power_3           peak_power_5        0.949292
peak_power_2           peak_power_3        0.942083
amplitude              sf_long             0.940658
magnitude_std          fourier_amp         0.933166
sf_short               max_visit_slope     0.924029
fourier_amp            sf_long             0.910773
magnitude_mean         median_error        0.910096
peak_power_2           peak_power_4        0.908188
amplitude              fourier_amp         0.904037

Complete predictor rows: 1915


## Export Results

In [15]:
FE_FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(FE_FEATURE_FILE, index=False)
period_validation.to_csv(FE_VALIDATION_FILE, index=False)
oracle.to_csv(FE_ORACLE_FILE, index=False)

## Final Checks

In [16]:
outputs = {
    "features": (FE_FEATURE_FILE, features),
    "validation": (FE_VALIDATION_FILE, period_validation),
    "oracle": (FE_ORACLE_FILE, oracle),
}

for name, (path, original) in outputs.items():
    saved = pd.read_csv(path)
    if len(saved) != len(original):
        raise ValueError(f"{name}: saved row count differs.")
    if list(saved.columns) != list(original.columns):
        raise ValueError(f"{name}: saved columns differ.")
    print(f"{name}: {len(saved)} rows, {len(saved.columns)} columns")

if set(period_validation["source_id"]) != set(sources["source_id"]):
    raise ValueError("Period validation source coverage is incomplete.")
if set(oracle["source_id"]) != set(stars.loc[stars["period_eligible"], "source_id"]):
    raise ValueError("Oracle source coverage is incomplete.")

features: 1978 rows, 58 columns
validation: 2000 rows, 46 columns
oracle: 1879 rows, 25 columns
